# Chia dữ liệu, chọn đặc trưng và khung huấn luyện/đánh giá dùng chung
Dữ liệu: `output/wine_quality_clean.csv` · Bài toán: phân loại 3 lớp `label` (0 = Thấp, 1 = Trung bình, 2 = Cao).

**Nội dung**
1. Chuẩn bị dữ liệu: mã hoá, kiểm soát rò rỉ, chia tập phân tầng, chọn 9 đặc trưng
2. Thiết kế thực nghiệm: baseline, Pipeline (chuẩn hoá chỉ fit trên train), Stratified 5-fold, chỉ số
3. Lưu dữ liệu và cấu hình dùng chung cho các notebook mô hình

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

OUT = Path('../output')
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5
N_FEATURES = 9
CLASS_NAMES = ['Thấp', 'Trung bình', 'Cao']

df = pd.read_csv(OUT / 'wine_quality_clean.csv')
print(df.shape)
df.head(3)

(5320, 15)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type,quality_group,label
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red,Thấp,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red,Thấp,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red,Thấp,0


## 1. Chuẩn bị dữ liệu
### 1.1 Mã hoá và kiểm soát rò rỉ
- `wine_type` mã hoá nhị phân (red = 0, white = 1). Phép ánh xạ cố định, không học từ dữ liệu nên không gây rò rỉ.
- Loại `quality`, `quality_group`: nhãn `label` được sinh trực tiếp từ hai cột này.
- Kiểm tra dòng trùng lặp: bản sao nằm ở cả train và test sẽ làm điểm đánh giá cao giả tạo.

In [2]:
PHYSICO = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
           'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
           'pH', 'sulphates', 'alcohol']
LEAK_COLS = ['quality', 'quality_group']   # nguồn sinh ra label
TARGET = 'label'

df['wine_type_white'] = (df['wine_type'] == 'white').astype(int)
ALL_FEATURES = PHYSICO + ['wine_type_white']

n_dup = df.duplicated(subset=ALL_FEATURES).sum()
print('Dòng trùng đặc trưng:', n_dup)
df = df.drop_duplicates(subset=ALL_FEATURES).reset_index(drop=True)

assert not set(LEAK_COLS + [TARGET]) & set(ALL_FEATURES)
assert df[ALL_FEATURES].isna().sum().sum() == 0

X, y = df[ALL_FEATURES], df[TARGET]
print('X:', X.shape)
print(y.value_counts(normalize=True).sort_index().round(3).to_dict())

Dòng trùng đặc trưng: 0
X: (5320, 12)
{0: 0.374, 1: 0.437, 2: 0.19}


### 1.2 Chia tập phân tầng
Chia 80/20 theo `label` **trước** mọi bước học từ dữ liệu (chọn đặc trưng, chuẩn hoá). Tập test chỉ dùng một lần ở bước đánh giá cuối.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

assert X_train.index.intersection(X_test.index).empty
print('Train:', X_train.shape, '| Test:', X_test.shape)
pd.DataFrame({'train': y_train.value_counts(normalize=True).sort_index(),
              'test': y_test.value_counts(normalize=True).sort_index()}).round(3)

Train: (4256, 12) | Test: (1064, 12)


,train,test
label,,
0,0.374,0.373
1,0.437,0.437
2,0.190,0.190


### 1.3 Chọn 9 đặc trưng (chỉ dùng tập train)
1. Loại `free sulfur dioxide` vì tương quan cao với `total sulfur dioxide` (đa cộng tuyến).
2. Xếp hạng các đặc trưng còn lại theo 3 tiêu chí: |tương quan| với nhãn, mutual information, độ quan trọng Random Forest.
3. Lấy hạng trung bình, giữ 9 đặc trưng tốt nhất.

In [4]:
corr = X_train.corr().abs()
pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(corr)
         for b in corr.columns[i+1:] if corr.loc[a, b] > 0.7]
print('Cặp tương quan > 0.7:', pairs)

Cặp tương quan > 0.7: [('free sulfur dioxide', 'total sulfur dioxide', np.float64(0.72))]


In [5]:
COLLINEAR_DROP = ['free sulfur dioxide']
cand = [c for c in ALL_FEATURES if c not in COLLINEAR_DROP]

scores = pd.DataFrame({
    'abs_corr': X_train[cand].corrwith(y_train).abs(),
    'mutual_info': mutual_info_classif(X_train[cand], y_train, random_state=RANDOM_STATE),
    'rf_importance': RandomForestClassifier(300, random_state=RANDOM_STATE, n_jobs=-1)
                         .fit(X_train[cand], y_train).feature_importances_,
})
scores['mean_rank'] = scores[['abs_corr', 'mutual_info', 'rf_importance']].rank(ascending=False).mean(axis=1)
scores = scores.sort_values('mean_rank')
SELECTED = scores.index[:N_FEATURES].tolist()
print('Đặc trưng được chọn:', SELECTED)
print('Bị loại:', [c for c in ALL_FEATURES if c not in SELECTED])
scores.round(3)

Đặc trưng được chọn: ['alcohol', 'density', 'volatile acidity', 'chlorides', 'total sulfur dioxide', 'citric acid', 'residual sugar', 'wine_type_white', 'sulphates']
Bị loại: ['fixed acidity', 'free sulfur dioxide', 'pH']


,abs_corr,mutual_info,rf_importance,mean_rank
alcohol,0.501,0.143,0.145,1.000
density,0.355,0.088,0.112,2.000
volatile acidity,0.260,0.030,0.110,3.667
chlorides,0.212,0.063,0.094,4.000
total sulfur dioxide,0.066,0.028,0.101,6.333
citric acid,0.091,0.032,0.087,6.333
residual sugar,0.077,0.017,0.088,7.000
wine_type_white,0.130,0.010,0.004,8.333
sulphates,0.040,0.007,0.093,8.667
pH,0.034,0.011,0.088,9.000


In [6]:
X_train_sel, X_test_sel = X_train[SELECTED], X_test[SELECTED]
assert len(SELECTED) == N_FEATURES

## 2. Thiết kế thực nghiệm
### 2.1 Khung dùng chung
- **Pipeline**: `StandardScaler` → mô hình. Khi đặt trong cross-validation, scaler chỉ fit trên phần train của từng fold, không rò rỉ sang fold validation.
- **Stratified 5-fold** (có xáo trộn, `random_state` cố định) trên tập train.
- **Chỉ số**: accuracy, balanced accuracy, macro-F1 (chỉ số chính, vì lớp "Cao" ít mẫu), weighted-F1.
- Tập test chỉ dùng qua `evaluate_on_test` khi đã chốt mô hình.

In [7]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
SCORING = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'f1_macro': 'f1_macro',
    'f1_weighted': 'f1_weighted',
}

def make_pipeline(model, scale=True):
    steps = [('scaler', StandardScaler())] if scale else []
    return Pipeline(steps + [('model', model)])

def run_cv(name, estimator, X=None, y=None):
    """Stratified K-fold trên train; trả về dict mean/std từng chỉ số."""
    X = X_train_sel if X is None else X
    y = y_train if y is None else y
    res = cross_validate(estimator, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    row = {'model': name}
    for m in SCORING:
        row[f'{m}_mean'] = res[f'test_{m}'].mean()
        row[f'{m}_std'] = res[f'test_{m}'].std()
    return row

def evaluate_on_test(estimator, X_tr=None, y_tr=None, X_te=None, y_te=None):
    """Fit trên toàn bộ train, đánh giá một lần trên test. Chỉ gọi khi đã chốt mô hình."""
    X_tr = X_train_sel if X_tr is None else X_tr
    y_tr = y_train if y_tr is None else y_tr
    X_te = X_test_sel if X_te is None else X_te
    y_te = y_test if y_te is None else y_te
    pred = estimator.fit(X_tr, y_tr).predict(X_te)
    print(classification_report(y_te, pred, target_names=CLASS_NAMES, digits=3))
    return {'accuracy': accuracy_score(y_te, pred),
            'balanced_accuracy': balanced_accuracy_score(y_te, pred),
            'f1_macro': f1_score(y_te, pred, average='macro'),
            'f1_weighted': f1_score(y_te, pred, average='weighted')}

### 2.2 Baseline
- `Dummy (most_frequent)` và `Dummy (stratified)`: mốc tối thiểu mọi mô hình phải vượt.
- `Logistic Regression`: baseline tuyến tính trong Pipeline.

In [8]:
baselines = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent'),
    'Dummy (stratified)': DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
    'Logistic Regression': make_pipeline(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
}
results = pd.DataFrame([run_cv(n, m) for n, m in baselines.items()]).set_index('model')
results[[c for c in results if c.endswith('_mean')]].round(3)

,accuracy_mean,balanced_accuracy_mean,f1_macro_mean,f1_weighted_mean
model,,,,
Dummy (most_frequent),0.437,0.333,0.203,0.265
Dummy (stratified),0.361,0.324,0.324,0.359
Logistic Regression,0.584,0.537,0.547,0.577


### 2.3 Kiểm tra rò rỉ của khung đánh giá
So sánh scaler fit **trong** Pipeline (đúng) với scaler fit trên toàn bộ train rồi mới CV (rò rỉ nhẹ). Phương án đúng là phương án dùng cho mọi mô hình sau này.

In [9]:
lr = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
X_prescaled = pd.DataFrame(StandardScaler().fit_transform(X_train_sel),
                           columns=SELECTED, index=X_train_sel.index)
leaky = run_cv('LR (scale trước CV - rò rỉ)', lr, X=X_prescaled)
proper = run_cv('LR (Pipeline - đúng)', make_pipeline(lr))
pd.DataFrame([leaky, proper]).set_index('model')[['f1_macro_mean', 'accuracy_mean']].round(4)

,f1_macro_mean,accuracy_mean
model,,
LR (scale trước CV - rò rỉ),0.5474,0.5841
LR (Pipeline - đúng),0.5474,0.5841


## 3. Lưu dữ liệu và cấu hình
Lưu dữ liệu **chưa chuẩn hoá**; các notebook mô hình dùng lại `make_pipeline`/`run_cv` để scaler luôn fit trong từng fold.

In [10]:
X_train_sel.to_csv(OUT / 'X_train.csv', index=False)
X_test_sel.to_csv(OUT / 'X_test.csv', index=False)
y_train.to_csv(OUT / 'y_train.csv', index=False)
y_test.to_csv(OUT / 'y_test.csv', index=False)

config = {'random_state': RANDOM_STATE, 'test_size': TEST_SIZE, 'n_splits': N_SPLITS,
          'features': SELECTED, 'target': TARGET, 'class_names': CLASS_NAMES,
          'main_metric': 'f1_macro'}
(OUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
results.round(4).to_csv(OUT / 'baseline_results.csv')
print('Đã lưu vào', OUT.resolve())

Đã lưu vào C:\Users\SuperCu\Desktop\ML-report\output
